### Chargement des données

In [ ]:
import numpy as np
import pandas as pd
import xml.etree.ElementTree as ET

try:
    import plotly.graph_objects as go
except ImportError:
    go = None

fichier = 4
if fichier == 1:
    nom_fichier = 'Daniel_Joram'
elif fichier == 2:
    nom_fichier = 'RDR2018 Alizés'
elif fichier == 3:
    nom_fichier = 'RDR2018 degolfage'
elif fichier == 4:
    nom_fichier = 'RDR2018 Sortie Manche'
elif fichier == 5:
    nom_fichier = 'Jef_Levasseur'
elif fichier == 6:
    nom_fichier = 'Francois_Rucheton'
elif fichier == 7:
    nom_fichier = 'Olivier_Costa'
elif fichier == 8:
    nom_fichier = 'Daniel_Joram_2020'
elif fichier == 9:
    nom_fichier = 'Tara_Ocean_Explorer'
elif fichier == 10:
    nom_fichier = 'merge'


data = pd.read_parquet(f'../Cleaned_Data/Cleaned_{nom_fichier}.parquet')

print(f'Nombre de données : {len(data)}')
data.head()


### Détection de voyages

In [ ]:
nb_voyages = data["voyage"].nunique()

voyages = []
for voyage_id in range(1, nb_voyages + 1):
    voyage_data = data[data["voyage"] == voyage_id]
    if len(voyage_data) < 2:
        continue
    start_idx = voyage_data.index[0]
    end_idx = voyage_data.index[-1]
    voyages.append((start_idx, end_idx))
print(f'Nombre de voyages détectés : {nb_voyages}')

### Statistiques

In [ ]:
print('True Wind ANgle (TWA) :')
print(data["TWA"].describe())

print('\nTrue Wind Speed (TWS) :')
print(data["TWS"].describe())

ratio = data['BSP']/data['TWS']
print('\nRatio BSP/TWS :')
print(ratio.describe())

# Possibilité détection moteur si ratio > x



### Affichage Polaire

In [ ]:
polaire = data[['TWA', 'TWS', 'BSP']].copy()
polaire.columns = ['TWA', 'TWS', 'BSP']

def tracer_heatmap_polaire(
    df,
    tws_range=None,
    angle_bin=2.0,
    speed_bin=0.3,
    max_speed=20,
    statistic="count"  # "count" ou "mean"
):
    """
    Heatmap polaire native Plotly via Barpolar
    """

    # ───────────── Filtrage TWS ─────────────
    if tws_range is not None:
        df = df[(df["TWS"] >= tws_range[0]) & (df["TWS"] < tws_range[1])]

    twa = df["TWA"].to_numpy()
    bsp = df["BSP"].to_numpy()

    # Symétrie tribord / babord
    twa = np.concatenate([twa, 360 - twa])
    bsp = np.tile(bsp, 2)

    # ───────────── Bins ─────────────
    angle_edges = np.arange(0, 360 + angle_bin, angle_bin)
    speed_edges = np.arange(0, max_speed + speed_bin, speed_bin)

    angle_idx = np.digitize(twa, angle_edges) - 1
    speed_idx = np.digitize(bsp, speed_edges) - 1

    valid = (
        (angle_idx >= 0) & (angle_idx < len(angle_edges) - 1) &
        (speed_idx >= 0) & (speed_idx < len(speed_edges) - 1)
    )

    angle_idx = angle_idx[valid]
    speed_idx = speed_idx[valid]
    bsp = bsp[valid]

    # ───────────── Agrégation ─────────────
    shape = (len(speed_edges)-1, len(angle_edges)-1)
    heatmap = np.zeros(shape)

    if statistic == "count":
        np.add.at(heatmap, (speed_idx, angle_idx), 1)
        z_label = "Densité"

    elif statistic == "mean":
        sums = np.zeros(shape)
        counts = np.zeros(shape)
        np.add.at(sums, (speed_idx, angle_idx), bsp)
        np.add.at(counts, (speed_idx, angle_idx), 1)
        heatmap = np.divide(sums, counts, where=counts > 0)
        z_label = "BSP moyenne"

    else:
        raise ValueError("statistic must be 'count' or 'mean'")

    heatmap = heatmap.astype(float)
    heatmap[heatmap == 0] = np.nan


    # ───────────── Conversion en Barpolar ─────────────
    theta_centers = angle_edges[:-1] + angle_bin / 2
    r_centers = speed_edges[:-1] + speed_bin / 2

    fig = go.Figure()

    for i, r in enumerate(r_centers):
        z = heatmap[i]

        fig.add_trace(go.Barpolar(
            r=np.full_like(theta_centers, speed_bin),
            theta=theta_centers,
            base=r - speed_bin / 2,
            width=np.full_like(theta_centers, angle_bin),
            marker=dict(
                color=z,
                colorscale="Turbo",
                cmin=0,
                cmax=np.nanmax(heatmap),
                showscale=False
            ),
            hoverinfo="skip"
        ))

    # Colorbar unique
    fig.add_trace(go.Barpolar(
        r=[0],
        theta=[0],
        marker=dict(
            color=[0],
            colorscale="Turbo",
            showscale=True,
            colorbar=dict(title=z_label)
        ),
        showlegend=False,
        hoverinfo="none"
    ))

    # ───────────── Layout polaire ─────────────
    fig.update_layout(
        title="Heatmap polaire (densité agrégée)",
        polar=dict(
            angularaxis=dict(
                direction="clockwise",
                rotation=90,
                tickmode="array",
                tickvals=list(range(0, 360, 30)),
                ticktext=[f"{i}°" for i in range(0, 360, 30)]
            ),
            radialaxis=dict(
                range=[0, max_speed],
                title="Vitesse fond (nds)"
            )
        ),
        height=800,
        showlegend=False
    )

    fig.show()

def tracer_polaire(df, tranches=None, step=5, rand_angle=1, legend_point_size=8):
    titre = 'Polaire de vent - toutes tranches de vent'
    if tranches is not None:
        titre = f'Polaire de vent - TWS {tranches[0][0]}-{tranches[-1][1]} nds'
    if tranches is None:
        tranches = [(0, 5), (5, 10), (10, 15), (15, 20), (20, 25), (25, 30), (30, 40), (40, 50)]

    couleurs = [
        '#1f77b4', '#ff7f0e', '#2ca02c',
        '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f'
    ]

    fig = go.Figure()

    # Sous-échantillonnage GLOBAL
    df = df.iloc[::step]

    for i, (tws_min, tws_max) in enumerate(tranches):
        m = (df['TWS'] >= tws_min) & (df['TWS'] < tws_max)
        d = df.loc[m, ['TWA', 'BSP']].to_numpy()

        if len(d) == 0:
            continue

        angles = np.deg2rad(d[:, 0])
        angles = np.concatenate([angles, 2*np.pi - angles])
        speeds = np.tile(d[:, 1], 2)

        angles_deg = np.rad2deg(angles)
        angles_deg += np.random.uniform(-rand_angle, rand_angle, size=len(angles_deg))
        angles_deg %= 360

        # Trace principale avec petits points
        fig.add_trace(go.Scatterpolar(
            r=speeds,
            theta=angles_deg,
            mode='markers',
            marker=dict(size=2, color=couleurs[i % len(couleurs)], opacity=0.5),
            name=f'TWS {tws_min}-{tws_max}',
            showlegend=False,
            legendgroup=f'group_{i}'
        ))

        # Trace légende avec gros points
        fig.add_trace(go.Scatterpolar(
            r=[None],
            theta=[None],
            mode='markers',
            marker=dict(size=legend_point_size, color=couleurs[i % len(couleurs)]),
            name=f'TWS {tws_min}-{tws_max}',
            legendgroup=f'group_{i}',
            hoverinfo='skip'
        ))

    fig.update_layout(
        title=titre,
        polar=dict(
            angularaxis=dict(direction='clockwise', rotation=90),
            radialaxis=dict(title='Vitesse (nds)')
        ),
        height=700
    )

    fig.show()

tracer_polaire(polaire)

# for i, v in enumerate(voyages):
#     polaire_voyage = polaire.iloc[v[0]:v[1]+1]
#     print(f'Voyage {i+1} :')
#     tracer_polaire(polaire_voyage)

# tracer_heatmap_polaire(polaire)


### Calcul distance et trajectoire

In [ ]:
def haversine_vec(lat1, lon1, lat2, lon2):
    """Haversine vectorisé (numpy arrays) — tableau de distances en mètres."""
    R = 6_371_000.0
    dlat = np.radians(lat2 - lat1)
    dlon = np.radians(lon2 - lon1)
    a = (
        np.sin(dlat / 2) ** 2
        + np.cos(np.radians(lat1))
        * np.cos(np.radians(lat2))
        * np.sin(dlon / 2) ** 2
    )
    return R * 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

def distance_parcourue(coords):
    lats = coords['latitude'].values
    lons = coords['longitude'].values
    return haversine_vec(lats[:-1], lons[:-1], lats[1:], lons[1:]).sum()

COULEURS_VOYAGES = [
    '#1f77b4', '#ff7f0e', '#2ca02c', '#d62728',
    '#9467bd', '#8c564b', '#e377c2', '#17becf',
    '#bcbd22', '#7f7f7f'
]

def get_coords_voyage(coords_all, voy):
    start, end = voy[0], voy[1]
    if end == -1:
        return coords_all.iloc[start:].reset_index(drop=True)
    return coords_all.iloc[start:end + 1].reset_index(drop=True)

def echantillonner_coords(coords_v, pas=5):
    """Retourne les indices : 1 point sur `pas`, depart et arrivee toujours inclus."""
    indices = list(range(0, len(coords_v), pas))
    if (len(coords_v) - 1) not in indices:
        indices.append(len(coords_v) - 1)
    return sorted(set(indices))

def afficher_tous_voyages(voyages_list, coords_all):
    if go is None:
        raise ImportError("plotly n'est pas installe. Installez-le avec: pip install plotly")

    all_traces = []
    traces_per_voyage = []
    all_lats = []
    all_lons = []

    for i, voy in enumerate(voyages_list):
        coords_v = get_coords_voyage(coords_all, voy)
        if len(coords_v) < 2:
            continue

        couleur = COULEURS_VOYAGES[i % len(COULEURS_VOYAGES)]
        indices = echantillonner_coords(coords_v)
        coords_echant = coords_v.iloc[indices]

        trace_start = len(all_traces)

        # Trace principale (ligne + points echantillonnes)
        all_traces.append(go.Scattermap(
            lat=coords_echant['latitude'].tolist(),
            lon=coords_echant['longitude'].tolist(),
            mode='lines+markers',
            name=f'Voyage {i + 1}',
            line=dict(color=couleur, width=2),
            marker=dict(size=5, color=couleur),
            hovertemplate=f'Voyage {i + 1}<br>Lat: %{{lat:.5f}}<br>Lon: %{{lon:.5f}}<extra></extra>',
            legendgroup=f'voyage_{i + 1}'
        ))
        # Marqueur depart
        all_traces.append(go.Scattermap(
            lat=[coords_v['latitude'].iloc[0]],
            lon=[coords_v['longitude'].iloc[0]],
            mode='markers',
            name=f'Depart V{i + 1}',
            marker=dict(size=12, color='green'),
            legendgroup=f'voyage_{i + 1}',
            showlegend=False
        ))
        # Marqueur arrivee
        all_traces.append(go.Scattermap(
            lat=[coords_v['latitude'].iloc[-1]],
            lon=[coords_v['longitude'].iloc[-1]],
            mode='markers',
            name=f'Arrivee V{i + 1}',
            marker=dict(size=12, color='red'),
            legendgroup=f'voyage_{i + 1}',
            showlegend=False
        ))
        traces_per_voyage.append(list(range(trace_start, len(all_traces))))
        all_lats.extend(coords_echant['latitude'].dropna().tolist())
        all_lons.extend(coords_echant['longitude'].dropna().tolist())

    total_traces = len(all_traces)
    n_voyages = len(traces_per_voyage)

    centre_lat = sum(all_lats) / len(all_lats) if all_lats else 0
    centre_lon = sum(all_lons) / len(all_lons) if all_lons else 0

    # Boutons du menu deroulant
    buttons = [dict(
        label='Tous les voyages',
        method='update',
        args=[{'visible': [True] * total_traces}]
    )]
    for i in range(n_voyages):
        visible = [False] * total_traces
        for idx in traces_per_voyage[i]:
            visible[idx] = True
        buttons.append(dict(
            label=f'Voyage {i + 1}',
            method='update',
            args=[{'visible': visible}]
        ))

    fig = go.Figure(data=all_traces)
    fig.update_layout(
        title='Trajectoires des voyages',
        template='plotly_white',
        height=700,
        map=dict(
            style='open-street-map',
            center=dict(lat=centre_lat, lon=centre_lon),
            zoom=4
        ),
        margin=dict(l=20, r=20, t=60, b=20),
        updatemenus=[dict(
            buttons=buttons,
            direction='down',
            showactive=True,
            x=0.01,
            xanchor='left',
            y=0.99,
            yanchor='top',
            bgcolor='white',
            bordercolor='gray'
        )],
        legend=dict(x=0.85, y=0.99)
    )
    fig.show()

coords = data[['longitude', 'latitude']].copy()

# Affichage des statistiques par voyage
if nb_voyages == 1:
    distance_totale_km = distance_parcourue(coords)/1000
    temps_total = (data['Date'].iloc[-1] - data['Date'].iloc[0]) / 3600
    if temps_total > 24:
        temps_total /= 24
        print(f'Temps total : {temps_total:.2f} jours')
    else:
        print(f'Temps total : {temps_total:.2f} heures')
    print(f'Distance totale parcourue : {distance_totale_km:.2f} km')
else:
    for i in range(nb_voyages - 1):
        print(f'Voyage {i + 1} :')
        coords_v = get_coords_voyage(coords, voyages[i])
        distance_totale_km = distance_parcourue(coords_v)/1000
        print(f'Distance totale parcourue : {distance_totale_km:.2f} km')
        temps_total = (data['Date'].iloc[voyages[i][1]] - data['Date'].iloc[voyages[i][0]]) / 3600
        if temps_total > 24:
            temps_total /= 24
            print(f'Temps total : {temps_total:.2f} jours')
        else:
            print(f'Temps total : {temps_total:.2f} heures')

afficher_tous_voyages(voyages, coords)


### Affichage polaire Temps

In [ ]:
def tracer_polaire_temps(
    df_source,
    nb_points=1000,
    pas=250,
    jitter_angle=1
):
    import numpy as np
    import pandas as pd
    import plotly.graph_objects as go

    if go is None:
        raise ImportError("plotly n'est pas installé. pip install plotly")

    # ───────────── Extraction & conversion du temps ─────────────
    dt = pd.to_datetime(df_source['Date'], unit="s", origin="unix")

    df = df_source[["TWA", "TWS", "BSP"]].copy()
    df["datetime"] = dt

    # ───────────── Nettoyage ─────────────
    df = df.dropna(subset=["datetime", "TWA", "TWS", "BSP"])
    df = df[
        (df["TWA"] >= 0) & (df["TWA"] <= 360) &
        (df["TWS"] > 0) &
        (df["BSP"] > 0)
    ].sort_values("datetime").reset_index(drop=True)

    if df.empty:
        print("Aucune donnée valide pour la polaire temporelle.")
        return

    total = len(df)
    fenetre = min(nb_points, total)
    pas = pas if pas > 0 else fenetre

    starts = list(range(0, max(total - fenetre + 1, 1), pas))
    if starts[-1] != total - fenetre:
        starts.append(total - fenetre)

    # ───────────── Trace optimisée ─────────────
    def creer_trace(sous_df, show_scale=False):
        theta = (
            sous_df["TWA"].to_numpy()
            + np.random.uniform(-jitter_angle, jitter_angle, size=len(sous_df))
        ) % 360

        return go.Scatterpolar(
            r=sous_df["BSP"],
            theta=theta,
            mode="markers",
            marker=dict(
                size=4,
                color=sous_df["TWS"],
                colorscale="Viridis",
                opacity=0.75,
                showscale=show_scale,
                colorbar=dict(title="TWS (nds)") if show_scale else None,
            ),
            hoverinfo="skip"
        )

    # ───────────── Frames + slider ─────────────
    frames = []
    slider_steps = []

    for i, start in enumerate(starts):
        sous_df = df.iloc[start:start + fenetre]

        dt_debut = sous_df["datetime"].iloc[0]
        dt_fin = sous_df["datetime"].iloc[-1]

        label = (
            f"{dt_debut.strftime('%d/%m %H:%M:%S')} "
            f"→ {dt_fin.strftime('%d/%m %H:%M:%S')}"
        )

        frames.append(go.Frame(
            name=str(i),
            data=[creer_trace(sous_df, show_scale=(i == 0))]
        ))

        slider_steps.append(dict(
            method="animate",
            args=[[str(i)], dict(
                mode="immediate",
                frame=dict(duration=0, redraw=True),
                transition=dict(duration=0)
            )],
            label=label
        ))

    # ───────────── Figure ─────────────
    fig = go.Figure(
        data=frames[0].data,
        frames=frames
    )

    fig.update_layout(
        title=f"Polaire temporelle – fenêtre glissante ({fenetre} points)",
        polar=dict(
            angularaxis=dict(
                direction="clockwise",
                rotation=90,
                tickmode="array",
                tickvals=list(range(0, 360, 30)),
                ticktext=[f"{i}°" for i in range(0, 360, 30)]
            ),
            radialaxis=dict(
                title="BSP (nds)",
                angle=135
            )
        ),
        sliders=[dict(
            active=0,
            currentvalue=dict(prefix="Période : "),
            pad=dict(t=40),
            steps=slider_steps
        )],
        updatemenus=[dict(
            type="buttons",
            direction="left",
            x=0.0,
            y=1.13,
            showactive=False,
            buttons=[
                dict(
                    label="Play",
                    method="animate",
                    args=[None, dict(
                        frame=dict(duration=300, redraw=True),
                        transition=dict(duration=0),
                        fromcurrent=True,
                        mode="immediate"
                    )]
                ),
                dict(
                    label="Pause",
                    method="animate",
                    args=[[None], dict(
                        frame=dict(duration=0, redraw=False),
                        transition=dict(duration=0),
                        mode="immediate"
                    )]
                )
            ]
        )],
        height=780,
        showlegend=False
    )

    fig.show()

# tracer_polaire_temps(data, nb_points=1000, pas=250)

### Polaire Tara

In [ ]:
def lire_polaire_xml(path_xml):
    """
    Lit un fichier XML de polaires et retourne :
    dict[TWS] = (angles_deg, vitesses)
    """
    tree = ET.parse(path_xml)
    root = tree.getroot()

    polaires = {}

    for curve in root.findall("PolarCurve"):
        tws = float(curve.find("PolarCurveIndex").attrib["value"])

        angles = []
        values = []

        for item in curve.findall("PolarItem"):
            angle = float(item.find("Angle").attrib["value"])
            value = float(item.find("Value").attrib["value"])

            # Ignore les zones nulles (0.1 = artificiel)
            if value <= 0.1:
                continue

            angles.append(angle)
            values.append(value)

        polaires[tws] = (np.array(angles), np.array(values))

    return polaires

def tracer_polaire_xml(polaires):
    fig = go.Figure()

    for tws in sorted(polaires.keys()):
        angles, speeds = polaires[tws]

        # Tribord
        theta_starboard = angles
        r_starboard = speeds

        # Babord (symétrie)
        theta_port = 360 - angles
        r_port = speeds

        # Courbe fermée (optionnel mais propre)
        theta_full = np.concatenate([
            theta_starboard,
            theta_port[::-1],
            [theta_starboard[0]]
        ])

        r_full = np.concatenate([
            r_starboard,
            r_port[::-1],
            [r_starboard[0]]
        ])

        fig.add_trace(go.Scatterpolar(
            theta=theta_full,
            r=r_full,
            mode="lines+markers",
            name=f"TWS {int(tws)} nds",
            line=dict(width=2),
            marker=dict(size=4)
        ))

    fig.update_layout(
        title="Polaire théorique (XML)",
        polar=dict(
            angularaxis=dict(
                direction="clockwise",
                rotation=90,
                tickmode="array",
                tickvals=list(range(0, 360, 30)),
                ticktext=[f"{i}°" for i in range(0, 360, 30)]
            ),
            radialaxis=dict(
                title="Vitesse bateau (nds)",
                angle=135
            )
        ),
        height=750,
        showlegend=True
    )

    fig.show()

polaire = lire_polaire_xml("Data/WindPolar Voile Stella-Maris.xml")
tracer_polaire_xml(polaire)